# Stage 4: PageIndex Builder Test Notebook

This notebook tests **Stage 4: The PageIndex Builder**, which creates a hierarchical navigation structure over documents—the equivalent of a "smart table of contents" that an LLM can traverse to locate information without reading the entire document.

## Complete Pipeline Flow

```
PDF Document
    ↓
Stage 1: Triage Agent → DocumentProfile
    ↓
Stage 2: Extraction Router → OCR Tools → ExtractedDocument
    ↓
Stage 3: Semantic Chunking Engine → List[LDU]
    ↓
Stage 4: PageIndex Builder ← Uses LDUs from Stage 3
    ↓
    PageIndex (hierarchical section tree)
```

**Key Point**: The PageIndex Builder takes the **chunked LDUs** from Stage 3 and builds a hierarchical navigation tree with:
- Section titles and page ranges
- Named entities per section
- LLM-generated summaries (2-3 sentences)
- Data types present (tables, figures, equations, etc.)

## Testing Focus

1. **Build PageIndex from LDUs** - Verify PageIndex tree structure
2. **Section Hierarchy** - Verify nested sections are correctly structured
3. **Section Enrichment** - Verify entities, summaries, and data types
4. **Topic-Based Search** - Test finding sections by topic
5. **Persistence** - Save and load PageIndex JSON
6. **Visualization** - Display the PageIndex tree structure

## Setup

In [1]:
import json
import os
from pathlib import Path
from typing import List

# Project imports
from src.agents.chunker import ChunkingEngine
from src.agents.indexer import PageIndexBuilder
from src.models.extracted_document import ExtractedDocument
from src.models.ldu import LDU
from src.models.page_index import PageIndex, Section

# Setup paths
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
PROFILES_DIR = BASE_DIR / ".refinery" / "profiles"
PAGEINDEX_DIR = BASE_DIR / ".refinery" / "pageindex"

# Create directories
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PAGEINDEX_DIR.mkdir(parents=True, exist_ok=True)

print("✓ Imports successful")
print(f"✓ Output directory: {OUTPUT_DIR}")
print(f"✓ PageIndex directory: {PAGEINDEX_DIR}")

c:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Imports successful
✓ Output directory: c:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\outputs
✓ PageIndex directory: c:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\.refinery\pageindex


## Step 1: Select Test Document

Select a PDF document and load its chunks (from Stage 3 output).

In [2]:
# Select a test document (modify as needed)
# Example: Class B - Scanned Document
pdf_path = DATA_DIR / "class_b" / "2018_Audited_Financial_Statement_Report.pdf"

# Validate PDF exists
if not pdf_path.exists():
    raise FileNotFoundError(f"PDF not found: {pdf_path}")

doc_id_base = pdf_path.stem  # Use filename without extension as base
print(f"✓ Selected document: {pdf_path.name}")
print(f"  Document ID base: {doc_id_base}")

✓ Selected document: 2018_Audited_Financial_Statement_Report.pdf
  Document ID base: 2018_Audited_Financial_Statement_Report


## Step 2: Load Chunks from Stage 3

Load the chunks (LDUs) that were created in Stage 3. If chunks don't exist, we can create them from the extracted document.

In [3]:
# Check for existing chunks from Stage 3
chunks_json_path = OUTPUT_DIR / f"{doc_id_base}_chunks.json"

if chunks_json_path.exists():
    print(f"📁 Loading chunks from: {chunks_json_path}")
    chunks_data = json.loads(chunks_json_path.read_text(encoding="utf-8"))
    chunks = [LDU(**chunk_data) for chunk_data in chunks_data]
    print(f"✓ Loaded {len(chunks)} chunks from Stage 3")
    print(f"  Chunk types: {set(c.chunk_type for c in chunks)}")
    print(f"  Total tokens: {sum(c.token_count for c in chunks):,}")
else:
    print(f"⚠️  Chunks not found at: {chunks_json_path}")
    print("  → Loading extracted document and chunking...")
    
    # Load extracted document
    extracted_json_path = OUTPUT_DIR / f"{doc_id_base}_extracted.json"
    if not extracted_json_path.exists():
        raise FileNotFoundError(
            f"Neither chunks nor extracted document found.\n"
            f"Please run Stage 2 (Extraction) and Stage 3 (Chunking) first.\n"
            f"Expected files:\n"
            f"  - {chunks_json_path}\n"
            f"  - {extracted_json_path}"
        )
    
    # Load and chunk
    extracted_doc_data = json.loads(extracted_json_path.read_text(encoding="utf-8"))
    extracted_doc = ExtractedDocument(**extracted_doc_data)
    
    chunking_engine = ChunkingEngine()
    chunks = chunking_engine.chunk(extracted_doc)
    
    # Save chunks for future use
    chunks_data = [chunk.model_dump() for chunk in chunks]
    chunks_json_path.write_text(json.dumps(chunks_data, indent=2), encoding="utf-8")
    print(f"✓ Created and saved {len(chunks)} chunks")

doc_id = doc_id_base  # Use base ID for PageIndex

📁 Loading chunks from: c:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\outputs\2018_Audited_Financial_Statement_Report_chunks.json
✓ Loaded 14 chunks from Stage 3
  Chunk types: {'table', 'paragraph', 'figure'}
  Total tokens: 2,420


## Step 3: Build PageIndex

Build the hierarchical PageIndex tree from the chunks.

In [4]:
# Initialize PageIndex Builder
# Set use_llm_summaries=False to use heuristic summaries (faster, no API calls)
# Set use_llm_summaries=True to generate LLM summaries (requires OPENROUTER_API_KEY)
use_llm = os.getenv("OPENROUTER_API_KEY") is not None

indexer = PageIndexBuilder(
    use_llm_summaries=use_llm,
    llm_model="gpt-4o-mini"  # Budget-friendly model
)

print("📑 Step 3: Building PageIndex from chunks...")
print(f"  Using LLM summaries: {use_llm}")
if use_llm:
    print(f"  LLM Model: {indexer.llm_model}")
else:
    print("  → Using heuristic summaries (set OPENROUTER_API_KEY for LLM summaries)")
print()

# Build PageIndex
page_index = indexer.build_from_ldus(
    doc_id=doc_id,
    doc_name=pdf_path.name,
    chunks=chunks
)

print(f"✓ PageIndex built successfully!")
print(f"  Document ID: {page_index.doc_id}")
print(f"  Document Name: {page_index.doc_name}")
print(f"  Root Sections: {len(page_index.root_sections)}")

# Count total sections (including nested)
def count_sections(sections: List[Section]) -> int:
    return len(sections) + sum(count_sections(s.child_sections) for s in sections)

total_sections = count_sections(page_index.root_sections)
print(f"  Total Sections (including nested): {total_sections}")

📑 Step 3: Building PageIndex from chunks...
  Using LLM summaries: True
  LLM Model: gpt-4o-mini



LLM summary generation failed: 402


✓ PageIndex built successfully!
  Document ID: 2018_Audited_Financial_Statement_Report
  Document Name: 2018_Audited_Financial_Statement_Report.pdf
  Root Sections: 1
  Total Sections (including nested): 1


## Step 4: Display PageIndex Structure

Visualize the hierarchical PageIndex tree.

In [5]:
def print_section_tree(section: Section, indent: int = 0, max_depth: int = 5) -> None:
    """Print section tree with indentation."""
    if indent > max_depth:
        return
    
    prefix = "  " * indent + "├─ " if indent > 0 else ""
    print(f"{prefix}{section.title}")
    print(f"{'  ' * (indent + 1)}Pages: {section.page_start}-{section.page_end}")
    
    if section.data_types_present:
        print(f"{'  ' * (indent + 1)}Data Types: {', '.join(section.data_types_present)}")
    
    if section.key_entities:
        entities_preview = ', '.join(section.key_entities[:5])
        if len(section.key_entities) > 5:
            entities_preview += f" ... (+{len(section.key_entities) - 5} more)"
        print(f"{'  ' * (indent + 1)}Entities: {entities_preview}")
    
    if section.summary:
        summary_preview = section.summary[:100] + "..." if len(section.summary) > 100 else section.summary
        print(f"{'  ' * (indent + 1)}Summary: {summary_preview}")
    
    print()
    
    # Print children
    for child in section.child_sections:
        print_section_tree(child, indent + 1, max_depth)

print("📊 PageIndex Structure:")
print("=" * 60)
print()

for root_section in page_index.root_sections:
    print_section_tree(root_section, indent=0, max_depth=4)

📊 PageIndex Structure:

Document Root
  Pages: 1-1
  Data Types: figure, table, text
  Entities: Amortizationintangibleassets, Birr, Borrowings, Capital, Cash ... (+5 more)
  Summary: Contains 14 content units including 5 tables, 8 figures



## Step 5: Test Section Enrichment

Verify that sections are properly enriched with entities, summaries, and data types.

In [6]:
print("🔍 Testing Section Enrichment:")
print("=" * 60)
print()

# Flatten all sections
def flatten_sections(sections: List[Section]) -> List[Section]:
    result = []
    for section in sections:
        result.append(section)
        result.extend(flatten_sections(section.child_sections))
    return result

all_sections = flatten_sections(page_index.root_sections)

print(f"Total sections: {len(all_sections)}")
print()

# Statistics
sections_with_entities = sum(1 for s in all_sections if s.key_entities)
sections_with_summaries = sum(1 for s in all_sections if s.summary)
sections_with_data_types = sum(1 for s in all_sections if s.data_types_present)

print(f"✓ Sections with entities: {sections_with_entities}/{len(all_sections)}")
print(f"✓ Sections with summaries: {sections_with_summaries}/{len(all_sections)}")
print(f"✓ Sections with data types: {sections_with_data_types}/{len(all_sections)}")
print()

# Show example enriched section
enriched_sections = [s for s in all_sections if s.key_entities or s.summary or s.data_types_present]
if enriched_sections:
    example = enriched_sections[0]
    print("📋 Example Enriched Section:")
    print(f"  Title: {example.title}")
    print(f"  Pages: {example.page_start}-{example.page_end}")
    print(f"  Data Types: {example.data_types_present}")
    print(f"  Entities ({len(example.key_entities)}): {example.key_entities[:5]}")
    print(f"  Summary: {example.summary}")
    print(f"  Child Sections: {len(example.child_sections)}")

🔍 Testing Section Enrichment:

Total sections: 1

✓ Sections with entities: 1/1
✓ Sections with summaries: 1/1
✓ Sections with data types: 1/1

📋 Example Enriched Section:
  Title: Document Root
  Pages: 1-1
  Data Types: ['figure', 'table', 'text']
  Entities (10): ['Amortizationintangibleassets', 'Birr', 'Borrowings', 'Capital', 'Cash']
  Summary: Contains 14 content units including 5 tables, 8 figures
  Child Sections: 0


## Step 6: Test Topic-Based Section Search

Test finding sections by topic keyword.

In [7]:
print("🔎 Testing Topic-Based Section Search:")
print("=" * 60)
print()

# Test queries
test_topics = [
    "financial",
    "table",
    "statement",
    "revenue",
    "expense"
]

for topic in test_topics:
    print(f"🔍 Searching for: '{topic}'")
    relevant_sections = indexer.find_sections_by_topic(
        page_index,
        topic=topic,
        top_k=3
    )
    
    if relevant_sections:
        for i, section in enumerate(relevant_sections, 1):
            print(f"  {i}. {section.title} (pages {section.page_start}-{section.page_end})")
    else:
        print("  No relevant sections found")
    print()

🔎 Testing Topic-Based Section Search:

🔍 Searching for: 'financial'
  No relevant sections found

🔍 Searching for: 'table'
  1. Document Root (pages 1-1)

🔍 Searching for: 'statement'
  No relevant sections found

🔍 Searching for: 'revenue'
  No relevant sections found

🔍 Searching for: 'expense'
  No relevant sections found



## Step 7: Save and Load PageIndex

Test persistence by saving and loading the PageIndex.

In [8]:
print("💾 Testing PageIndex Persistence:")
print("=" * 60)
print()

# Save PageIndex
saved_path = indexer.save_page_index(page_index, PAGEINDEX_DIR)
print(f"✓ Saved PageIndex to: {saved_path}")
print(f"  File size: {saved_path.stat().st_size / 1024:.2f} KB")
print()

# Load PageIndex
loaded_page_index = indexer.load_page_index(saved_path)
print(f"✓ Loaded PageIndex from: {saved_path}")
print(f"  Document ID: {loaded_page_index.doc_id}")
print(f"  Document Name: {loaded_page_index.doc_name}")
print(f"  Root Sections: {len(loaded_page_index.root_sections)}")

# Verify data integrity
assert loaded_page_index.doc_id == page_index.doc_id
assert loaded_page_index.doc_name == page_index.doc_name
assert len(loaded_page_index.root_sections) == len(page_index.root_sections)
print("\n✓ Data integrity verified")

💾 Testing PageIndex Persistence:

✓ Saved PageIndex to: c:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\.refinery\pageindex\2018_Audited_Financial_Statement_Report_pageindex.json
  File size: 0.74 KB

✓ Loaded PageIndex from: c:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\.refinery\pageindex\2018_Audited_Financial_Statement_Report_pageindex.json
  Document ID: 2018_Audited_Financial_Statement_Report
  Document Name: 2018_Audited_Financial_Statement_Report.pdf
  Root Sections: 1

✓ Data integrity verified


## Step 8: Export PageIndex to Markdown

Export the PageIndex structure to a human-readable Markdown format.

In [9]:
def pageindex_to_markdown(page_index: PageIndex) -> str:
    """Convert PageIndex to Markdown format."""
    lines = [f"# PageIndex: {page_index.doc_name}"]
    lines.append(f"\n**Document ID:** {page_index.doc_id}")
    lines.append(f"**Total Root Sections:** {len(page_index.root_sections)}")
    lines.append("\n---\n")
    
    def section_to_markdown(section: Section, level: int = 1) -> List[str]:
        """Convert a section to Markdown."""
        result = []
        
        # Section header
        header_level = "#" * (level + 1)
        result.append(f"{header_level} {section.title}")
        result.append(f"\n**Pages:** {section.page_start}-{section.page_end}")
        
        # Data types
        if section.data_types_present:
            result.append(f"\n**Data Types:** {', '.join(section.data_types_present)}")
        
        # Entities
        if section.key_entities:
            entities_str = ', '.join(section.key_entities[:10])
            if len(section.key_entities) > 10:
                entities_str += f" ... (+{len(section.key_entities) - 10} more)"
            result.append(f"\n**Key Entities:** {entities_str}")
        
        # Summary
        if section.summary:
            result.append(f"\n**Summary:** {section.summary}")
        
        result.append("\n")
        
        # Child sections
        for child in section.child_sections:
            result.extend(section_to_markdown(child, level + 1))
        
        return result
    
    # Convert all root sections
    for root_section in page_index.root_sections:
        lines.extend(section_to_markdown(root_section, level=1))
    
    return "\n".join(lines)

# Export to Markdown
markdown_content = pageindex_to_markdown(page_index)
markdown_path = OUTPUT_DIR / f"{doc_id}_pageindex.md"
markdown_path.write_text(markdown_content, encoding="utf-8")

print(f"✓ Exported PageIndex to Markdown: {markdown_path}")
print(f"  Preview (first 500 chars):")
print("-" * 60)
print(markdown_content[:500] + "..." if len(markdown_content) > 500 else markdown_content)

✓ Exported PageIndex to Markdown: c:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\outputs\2018_Audited_Financial_Statement_Report_pageindex.md
  Preview (first 500 chars):
------------------------------------------------------------
# PageIndex: 2018_Audited_Financial_Statement_Report.pdf

**Document ID:** 2018_Audited_Financial_Statement_Report
**Total Root Sections:** 1

---

## Document Root

**Pages:** 1-1

**Data Types:** figure, table, text

**Key Entities:** Amortizationintangibleassets, Birr, Borrowings, Capital, Cash, Cashflowsfrom, Cashflowsfromfinancingactivities, Cashflowsfromoperatingactivities, Cashgeneratedfrom, Contract

**Summary:** Contains 14 content units including 5 tables, 8 figures




In [10]:
# Create comprehensive summary
summary = {
    "document": {
        "doc_id": page_index.doc_id,
        "doc_name": page_index.doc_name,
    },
    "pageindex": {
        "root_sections": len(page_index.root_sections),
        "total_sections": total_sections,
        "page_range": {
            "start": min((s.page_start for s in flatten_sections(page_index.root_sections)), default=1),
            "end": max((s.page_end for s in flatten_sections(page_index.root_sections)), default=1),
        },
    },
    "enrichment": {
        "sections_with_entities": sections_with_entities,
        "sections_with_summaries": sections_with_summaries,
        "sections_with_data_types": sections_with_data_types,
        "total_entities": sum(len(s.key_entities) for s in flatten_sections(page_index.root_sections)),
    },
    "output_files": {
        "pageindex_json": str(saved_path),
        "pageindex_markdown": str(markdown_path),
    },
}

# Save summary
summary_path = OUTPUT_DIR / f"{doc_id}_pageindex_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("📊 PageIndex Builder Test Summary")
print("=" * 60)
print(f"\nDocument: {page_index.doc_name}")
print(f"\nPageIndex Structure:")
print(f"  Root Sections: {summary['pageindex']['root_sections']}")
print(f"  Total Sections: {summary['pageindex']['total_sections']}")
print(f"  Page Range: {summary['pageindex']['page_range']['start']}-{summary['pageindex']['page_range']['end']}")
print(f"\nEnrichment:")
print(f"  Sections with Entities: {summary['enrichment']['sections_with_entities']}")
print(f"  Sections with Summaries: {summary['enrichment']['sections_with_summaries']}")
print(f"  Sections with Data Types: {summary['enrichment']['sections_with_data_types']}")
print(f"  Total Entities Extracted: {summary['enrichment']['total_entities']}")
print(f"\nOutput Files:")
print(f"  JSON: {saved_path.name}")
print(f"  Markdown: {markdown_path.name}")
print(f"  Summary: {summary_path.name}")
print(f"\n✓ Complete analysis saved to: {summary_path}")

📊 PageIndex Builder Test Summary

Document: 2018_Audited_Financial_Statement_Report.pdf

PageIndex Structure:
  Root Sections: 1
  Total Sections: 1
  Page Range: 1-1

Enrichment:
  Sections with Entities: 1
  Sections with Summaries: 1
  Sections with Data Types: 1
  Total Entities Extracted: 10

Output Files:
  JSON: 2018_Audited_Financial_Statement_Report_pageindex.json
  Markdown: 2018_Audited_Financial_Statement_Report_pageindex.md
  Summary: 2018_Audited_Financial_Statement_Report_pageindex_summary.json

✓ Complete analysis saved to: c:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\outputs\2018_Audited_Financial_Statement_Report_pageindex_summary.json
